# Full pipeline: from microscope images to lineage

For one experiment, this notebook does everything that used to be done with commands in the terminal:

1. **Aligned stacks**: gathers the individual images of each position into a stack and aligns it using the numbers on the chip (`generar_stacks.py`). If a position has two different fields saved under the same number, it splits them into `<pos>` and `<pos>b`. It then checks the alignment frame by frame (`residuo_px` in the registration table).
2. **Straightened channels**: detects the channels, makes the kymographs and a stack with each straightened channel (`canales_enderezados.py`, `kimogramas.py`).
3. **Cells and lineage**: the centre-based detector (neural network) finds the cells in each frame, and their boxes are tracked by solving the whole movie at once, including isolated cells below a gap, with their divisions, exits and deaths (`detectar_celulas.py predecir`, `linaje.py`).
4. **Stacks with boxes**: the aligned stack with a coloured box and a number for each cell (`stack_linaje.py`).
5. **Summary**: a table per position and the growth rate over time.
6. **Mothers' Excel file**: one sheet per position with the length of the cell at the closed end of each channel in each frame, to be checked and completed by hand (`excel_madres.py`).
7. **Fluorescence per cell** (if there are fluorescence images): the fluorescence of every cell of the lineage in every frame (`fluorescencia_linaje.py`).

**How to use it:** edit the *Parameters* cell and run everything (*Run All*). Each step skips what has already been done, so if it is interrupted it can be run again without losing previous work. To redo something, set `REHACER = True`.

This notebook must be in the `scripts` folder, next to the programs. See `docs/usage.md` for the image naming convention, the outputs and the meaning of every column.

## Parameters

In [ ]:
from pathlib import Path

CARPETA_IMAGENES = Path('/path/to/MyExperiment/Images')   # the individual microscope images of ONE experiment;
                                          # results go to the sibling folder MyExperiment/Stacks
POSICIONES = None                         # None = all; or a list, e.g. ['01', '02']
SOLO_BF = None                            # None = detect automatically; True if there are no fluorescence images
FLUO = 'PI'                               # fluorescence channel(s): 'PI', 'GFP'… or a list from the brightest to the
                                          # dimmest, e.g. ['GFP', 'PI'] (stacks <pos>-<date>-GFP.tif, -PI.tif)
REHACER = False                           # True = redo all steps even if their results already exist
PASOS = {'stacks': True, 'canales': True, 'celulas': True, 'cajas': True, 'resumen': True, 'excel': True, 'fluorescencia': True}

In [ ]:
import sys, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

faltan = []
for mod, pip in [('tifffile', 'tifffile'), ('skimage', 'scikit-image'), ('scipy', 'scipy'), ('roifile', 'roifile'),
                 ('PIL', 'pillow'), ('joblib', 'joblib'), ('openpyxl', 'openpyxl'), ('sklearn', 'scikit-learn==1.7.2')]:
    try: __import__(mod)
    except ImportError: faltan.append(pip)
if faltan:
    raise SystemExit('Missing packages. Run in a cell:  %pip install ' + ' '.join(faltan))
try:
    import onnxruntime
except ImportError:
    print('Warning: onnxruntime is missing, so the old (tree-based) detector will be used instead of the network (centre-based detector). '
          'To use the network, run in a cell:  %pip install onnxruntime  and restart the kernel.')
import sklearn
if sklearn.__version__ != '1.7.2':
    print(f'Warning: you have scikit-learn {sklearn.__version__} and the detector was saved with 1.7.2. If loading it fails '
          '(step 3), run in a cell:  %pip install scikit-learn==1.7.2  and restart the kernel.')

SCRIPTS = Path.cwd()
if not (SCRIPTS / 'detectar_celulas.py').exists():
    raise SystemExit('Open this notebook from the scripts folder (where detectar_celulas.py is).')
sys.path.insert(0, str(SCRIPTS))
import importlib
for _m in ('alinear_stack', 'centros_comun', 'linaje', 'segmentar_kimogramas', 'kimogramas', 'leer_cajas',
           'generar_stacks', 'canales_enderezados', 'detectar_celulas', 'stack_linaje', 'excel_madres', 'fluorescencia_linaje'):
    if _m in sys.modules: importlib.reload(sys.modules[_m])   # if the kernel had already loaded them, use the current version
import generar_stacks as GS, canales_enderezados as CE, detectar_celulas as DC, stack_linaje as SL, excel_madres as EM
import fluorescencia_linaje as FL
get_ipython().run_line_magic('matplotlib', 'inline')   # some programs set a windowless matplotlib backend (Agg): here, figures go in the notebook
import joblib

CARPETA_IMAGENES = Path(CARPETA_IMAGENES).resolve()
STACKS = CARPETA_IMAGENES.parent / 'Stacks'          # results next to the image folder: <experiment>/Stacks
FLUOS = [FLUO] if isinstance(FLUO, str) else list(FLUO)
CANALES = STACKS / 'Canales'; LINAJE = STACKS / 'Linaje'
print('Images :', CARPETA_IMAGENES)
print('Results:', STACKS)
if not CARPETA_IMAGENES.exists():
    padre = CARPETA_IMAGENES.parent
    print(f'\n{CARPETA_IMAGENES} does not exist.')
    if padre.exists():
        print(f'Contents of {padre}:', sorted(p.name for p in padre.iterdir())[:30])
    raise SystemExit('Change CARPETA_IMAGENES in the parameters cell.')

## 0. Available images

The images must be named like those from our usual microscope: `<position>_<yymmdd>_<hhmmss>_anything.tif`, for example `04_260925_103958_Regrowth_BF_00300_.tif`. The position and the actual time of each frame, which is used to measure growth, are taken from the name.

In [ ]:
try:
    inv, umbral = GS.inventario(CARPETA_IMAGENES, FLUOS)
except FileNotFoundError:
    inv = pd.DataFrame()
todas = sorted(p.name for p in CARPETA_IMAGENES.glob('*.tif*'))
if len(inv) == 0:
    print(f'There are {len(todas)} .tif files, but none has the expected name. Examples:', todas[:5])
    raise SystemExit('Rename the images (position_yymmdd_hhmmss_….tif) or adapt the file-name pattern in generar_stacks.py.')
if len(inv) < len(todas):
    print(f'Warning: {len(todas) - len(inv)} of {len(todas)} files do not have the expected name and are ignored.')
display(inv.groupby('pos').agg(imagenes=('archivo', 'size'), primera=('hora', 'min'), ultima=('hora', 'max')))
pi = GS.hay_pi(inv) if SOLO_BF is None else not SOLO_BF
print(f'Paired fluorescence images ({", ".join(FLUOS)}):', 'yes' if pi else 'no (bright field only)')
if POSICIONES:
    print('Only processing positions', POSICIONES)

## 1. Aligned stacks

One stack per position (`<pos>-<ddmmyyyy>-BF.tif` and, if there is fluorescence, `-<FLUO>.tif`), aligned using the numbers on the chip. The alignment *quality* is the correlation with the first frame (1 = perfect); below 0.8 it is worth looking at the stack.

First it checks that each frame belongs to its position: if in some round the positions (areas) were saved with shifted numbers (area 11 as 10, 12 as 11…), each frame is compared with the chip numbers of its position and of the neighbouring ones and moved to the position it belongs to (the changes are listed in `Stacks/areas_reasignadas.csv`). This takes ~2–3 min the first time with ~1300 frames; afterwards the result is reused.

In [ ]:
def hechos_bf():
    return {p.name.split('-')[0]: p for p in sorted(STACKS.glob('*-BF.tif'))}

if PASOS['stacks']:
    pos_todas = sorted(inv.pos.unique()) if not POSICIONES else list(POSICIONES)
    faltan = pos_todas if REHACER else [p for p in pos_todas if p not in hechos_bf()]
    if faltan:
        t0 = time.time()
        GS.generar_stacks(CARPETA_IMAGENES, salida=STACKS, posiciones=faltan, solo_bf=SOLO_BF, fluo=FLUO)
        print(f'{len(faltan)} positions in {time.time() - t0:.0f} s')
    else:
        print('Stacks already done (REHACER = False).')
BF = {p: r for p, r in hechos_bf().items() if not POSICIONES or p in POSICIONES}
print(len(BF), 'stacks:', ', '.join(r.name for r in BF.values()))
for p, r in BF.items():
    reg = r.with_name(r.name.replace('-BF.tif', '_registro.csv'))
    if reg.exists():
        t = pd.read_csv(reg); col = [c for c in t.columns if 'calidad' in c]
        if col:
            q = t[col[0]]; malas = int((q < 0.8).sum())
            dud = t.slice[(t.residuo_px > 2) | (t.encaje_anterior < 0.8)].tolist() if 'residuo_px' in t else []
            print(f'  {p}: {len(t)} frames, minimum quality {q.min():.2f}' + (f' — {malas} frames with quality < 0.8: check them' if malas else '')
                  + (f' — QUESTIONABLE ALIGNMENT in slices {dud} (relative to the previous one)' if dud else ''))

In [ ]:
# first and last frame of the first position (to check that it is well aligned)
import tifffile
if BF:
    p, r = next(iter(BF.items()))
    with tifffile.TiffFile(r) as tf:
        a, b = tf.pages[0].asarray(), tf.pages[len(tf.pages) - 1].asarray()
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for x, im, t in zip(ax, (a, b), ('first frame', 'last frame')):
        x.imshow(im, cmap='gray'); x.set_title(f'{r.name}: {t}'); x.axis('off')
    plt.tight_layout(); plt.show()

## 2. Straightened channels and kymographs

Detects the channels (numbered from left to right), makes the kymographs (`Stacks/Kimogramas/`) and a stack with each straightened channel (`Stacks/Canales/<pos>_canales_BF.tif`). That stack is the one you open in Fiji with `Cajas_Canales.ijm` to check the boxes with [8].

In [ ]:
if PASOS['canales']:
    for p, r in BF.items():
        base = r.stem.replace('-BF', '')
        ce = CANALES / f'{base}_canales_BF.tif'
        hecho = ce.exists() and ce.stat().st_mtime > r.stat().st_mtime    # redone if the stack is newer
        if hecho and not REHACER:
            print(f'{p}: already done'); continue
        t0 = time.time(); CE.canales_enderezados(r); print(f'   ({time.time() - t0:.0f} s)')
geos = {p: pd.read_csv(CANALES / f"{r.stem.replace('-BF', '')}_canales_BF_geometria.csv") for p, r in BF.items()
        if (CANALES / f"{r.stem.replace('-BF', '')}_canales_BF_geometria.csv").exists()}
display(pd.DataFrame({p: {'canales': len(g), 'fotos': int(g.fotos.iloc[0]), 'en el borde de la imagen': int(g.al_borde.sum())}
                      for p, g in geos.items()}).T)
if BF:
    png = STACKS / 'Kimogramas' / f"{next(iter(BF.values())).stem.replace('-BF', '')}_kimogramas.png"
    if png.exists(): display(Image(filename=str(png), width=900))

## 3. Cells and lineage

For each position: the centre-based detector (neural network, `modelos/detector_centros.onnx`) finds the cells in each frame of each channel. Their boxes are tracked by solving the whole movie of each channel at once (integer programming), keeping the order in the channel (cells do not overtake each other) and allowing gaps between cells: each cell can continue, divide, leave through the exit or disappear inside the channel (death, lysis or loss), and the network may miss a cell in some frame (its box is filled in, dotted in the stack) or see a box that is not a cell. It also works with filaments and with dying cells. It takes 1–2 minutes per position (`detectar_celulas.SEGUIMIENTO = 'foto_a_foto'` is faster and somewhat worse). If `onnxruntime` is missing, the old (tree-based) detector is used, with the cascade from the closed end, which is slower, worse and does not track isolated cells.

**Frames corrected by hand:** in the frames of a channel in which you have corrected or drawn the boxes with `Cajas_Canales.ijm` (those stored in `<pos>_canales_BF_cajas.zip`), the tracking uses your boxes instead of the network's; in the other frames, it uses the network's. Your boxes are mandatory: each one is a cell, they are never merged or discarded, and their length is always measured (unless the cell goes past the exit). Within the zone you annotated (down to your last box in that frame; the whole channel if you marked it as empty), a cell you did not draw a box for was not in that frame. The boxes proposed by [8] are still the network's. In `_linaje.csv`, the `manual` column says which boxes are yours.

A position is redone if its lineage is older than the image, your boxes (`_cajas.zip`), the programs (`detectar_celulas.py`, `linaje.py`) or the network (`modelos/detector_centros.onnx`), so after correcting boxes, updating the programs or retraining, just run it again. It writes to `Stacks/Canales/`:

- `<pos>_canales_BF_detector.txt`: the boxes proposed by [8]; in yellow, those with confidence < 0.6.
- `_revisar.txt`: the uncertain frames, for [9].
- `_linaje.csv`: each cell in each frame (`interpolada` = box filled in for a frame in which the network did not see the cell; no length).
- `_linaje_ciclos.csv`: one row per cell, with how it ends (`final`: divides, exits, disappears inside the channel or end of the movie; the values are in Spanish: `se divide`, `sale o deja de verse`, `desaparece en el canal…`, `fin de la película`).
- `_crecimiento_posicion.csv`: the growth rate of the position, frame by frame.

In [ ]:
if PASOS['celulas']:
    try:
        import onnxruntime  # noqa
        print('Detector: network (centre-based detector,', DC.Path(DC.__file__).parent / 'modelos' / 'detector_centros.onnx', ')')
        modelo = None
    except ImportError:
        modelo = joblib.load(DC.MODELO); print('Detector: tree-based,', DC.MODELO.name)
    t_prog = max(Path(DC.__file__).stat().st_mtime, (Path(DC.__file__).parent / 'linaje.py').stat().st_mtime,
                 (Path(DC.__file__).parent / 'modelos' / 'detector_centros.onnx').stat().st_mtime)
    for p, r in BF.items():
        img = CANALES / f"{r.stem.replace('-BF', '')}_canales_BF.tif"
        lin = img.with_name(img.stem + '_linaje.csv')
        zp = img.with_name(img.stem + '_cajas.zip')                 # your boxes: if you change them, it is redone
        t_cajas = zp.stat().st_mtime if zp.exists() else 0
        if lin.exists() and lin.stat().st_mtime > max(img.stat().st_mtime, t_prog, t_cajas) and not REHACER:
            print(f'{p}: already done'); continue
        t0 = time.time()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore'); DC.predecir(img, modelo)
        print(f'   ({time.time() - t0:.0f} s)')

## 4. Stacks with one box per cell

`Stacks/Linaje/<pos>-<ddmmyyyy>-BF_linaje.tif`: the aligned stack in colour, with one box per cell. Each cell has its own colour and number (the `celula` of the tables, unique within each channel); when a cell divides, the daughters get a new number and colour. Dashed boxes are cells that reach the channel exit; dotted ones are frames in which the network did not see the cell and the tracking filled in its box. Each frame has its number and time written at the top. The stacks are losslessly compressed (they take ~2.7 times less space): in Fiji, open them with *File › Import › Bio-Formats* (or by dragging the file onto Fiji).

In [ ]:
if PASOS['cajas']:
    for p, r in BF.items():
        dest = LINAJE / f'{r.stem}_linaje.tif'
        lin = CANALES / f"{r.stem.replace('-BF', '')}_canales_BF_linaje.csv"
        if not lin.exists(): print(f'{p}: lineage missing (step 3)'); continue
        if dest.exists() and dest.stat().st_mtime > lin.stat().st_mtime and not REHACER:
            print(f'{p}: already done'); continue
        SL.stack_linaje(r, canales=CANALES, salida=LINAJE)

In [ ]:
# a frame from the middle of the movie of the first position, to see the boxes
if BF:
    p, r = next(iter(BF.items())); dest = LINAJE / f'{r.stem}_linaje.tif'
    if dest.exists():
        with tifffile.TiffFile(dest) as tf:
            n = len(tf.pages); im = tf.pages[n // 2].asarray()
        plt.figure(figsize=(13, 9)); plt.imshow(im); plt.axis('off'); plt.title(f'{dest.name}: frame {n // 2 + 1}'); plt.show()

## 5. Summary

Cycles that pass the same quality control as in the LB analysis:

- born and divided within the movie;
- born from a symmetric division;
- inherit 35–65 % of their mother's length;
- Ld/L0 between 1.4 and 2.8.

The table also counts divisions, cells that disappear inside the channel (death or lysis; in the first frames, also those washed away by the flow when the chip is loaded) and filaments (cells that grow to more than 4 times the typical birth length without dividing).

Lengths are in pixels and the rate is in 1/min, using the actual time of each frame.

In [ ]:
if PASOS['resumen']:
    filas, Ks = [], []
    for p, r in BF.items():
        base = r.stem.replace('-BF', '')
        f = CANALES / f'{base}_canales_BF_linaje_ciclos.csv'
        if not f.exists(): continue
        K = pd.read_csv(f).assign(pos=p); Ks.append(K)
        F = pd.read_csv(CANALES / f'{base}_canales_BF_linaje.csv')
        ok = (K.completo.astype(bool) & K.nace_por.eq('simétrica') & K.reparto.between(0.35, 0.65)
              & (K.Ld_px / K.L0_px).between(1.4, 2.8))
        q = K[ok]
        vis = F.groupby('canal').visible.mean() if 'visible' in F else pd.Series(dtype=float)
        l0 = K.L0_px.median() if K.L0_px.notna().any() else 25.0
        Lmax = F[F.visible.astype(bool)].groupby(['canal', 'celula']).longitud_px.max() if 'visible' in F else F.groupby(['canal', 'celula']).longitud_px.max()
        fin = K.final.astype(str)
        filas.append({'posición': p, 'canales con células': int((vis > 0.05).sum()), 'células': len(K),
                      'divisiones': int(fin.str.startswith('se divide').sum()),
                      'desaparecen': int(fin.str.startswith('desaparece').sum()),
                      'filamentos': int((Lmax > 4 * l0).sum()),
                      'ciclos válidos': len(q), 'duración (min)': q.duracion_min.median(),
                      'L al nacer (px)': q.L0_px.median(), 'L al dividirse (px)': q.Ld_px.median(),
                      'tasa (1/min)': q.tasa_por_min.median()})
    T = pd.DataFrame(filas).set_index('posición')
    display(T.round(3))
    if Ks:
        pd.concat(Ks).to_csv(STACKS / 'linaje_ciclos_todas.csv', index=False)
        print('All cells from all positions:', STACKS / 'linaje_ciclos_todas.csv')
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for p, r in BF.items():
        f = CANALES / f"{r.stem.replace('-BF', '')}_canales_BF_crecimiento_posicion.csv"
        if f.exists():
            g = pd.read_csv(f); ax.plot(g.tiempo_min, g.tasa_esperada_por_min, lw=1.5, label=p)
    ax.set(xlabel='time (min)', ylabel='growth rate (1/min)', title='Growth rate of each position')
    ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2, frameon=False); plt.tight_layout(); plt.show()

## 6. Excel file with the mothers' length

`Stacks/Excel/<pos>-<ddmmyyyy>_longitud_madres.xlsx`, one sheet per position:

- row 1: time of each frame; row 2: frame number in the stack;
- two rows per channel (column A: channel number, the same one drawn in the Linaje stacks): the upper row is the length (px) of the cell at the closed end of the channel (old pole) in each frame; the lower row, only in the first frame in which the mother's division is seen, is the daughter's length (mother + daughter in that frame = the mother before dividing);
- NaN in red: the cell goes past the exit (to be measured by hand); in yellow: the cell is there but there is no reliable length (filled-in box, or a box that does not fit); in grey: bad frame; in green: lengths from frames corrected by hand.

It is redone if the lineage is newer. Careful: if you write measurements into the Excel file, save it under another name so that it is not overwritten when you run the notebook again.

In [ ]:
if PASOS['excel']:
    for p, r in BF.items():
        base = r.stem.replace('-BF', '')
        lin = CANALES / f'{base}_canales_BF_linaje.csv'
        dest = STACKS / 'Excel' / f'{base}_longitud_madres.xlsx'
        if not lin.exists(): print(f'{p}: lineage missing (step 3)'); continue
        if dest.exists() and dest.stat().st_mtime > lin.stat().st_mtime and not REHACER:
            print(f'{p}: already done'); continue
        EM.excel_madres(r, canales=CANALES)

## 7. Fluorescence of each cell

For each fluorescence stack (`<pos>-<ddmmyyyy>-<FLUO>.tif`), the channels are straightened with the same geometry as the bright field and, for every cell of the lineage in every frame, the mean intensity inside its box is measured, minus the background of the channel walls and of the medium (`fluo_neta`, proportional to the concentration; `fluo_total` = `fluo_neta` × length). Frames without a fluorescence image are not measured. Writes `Stacks/Canales/<pos>_canales_BF_linaje_<FLUO>.csv`. It is redone when the lineage is newer.

In [ ]:
if PASOS['fluorescencia']:
    for p, r in BF.items():
        base = r.stem.replace('-BF', '')
        lin = CANALES / f'{base}_canales_BF_linaje.csv'
        for f in FLUOS:
            st = r.with_name(f'{base}-{f}.tif')
            if not st.exists() or not lin.exists(): continue
            dest = CANALES / f'{base}_canales_BF_linaje_{f}.csv'
            if dest.exists() and dest.stat().st_mtime > lin.stat().st_mtime and not REHACER:
                print(f'{p} {f}: already done'); continue
            FL.fluorescencia_linaje(r, fluo=f)

## What to check before trusting the numbers

- **Alignment:** if any position has frames with quality < 0.8, look at its stack in Fiji.
- **Channels:** all channels must appear in the kymograph figure, with the closed end and the exit in the right place.
- **Boxes:** open a couple of `Stacks/Linaje/*_linaje.tif` files in Fiji (with Bio-Formats) and step through the frames. The detector was trained on our images. If the images of this experiment are different (another magnification, chip, strain or focus), it may fail. In that case, check a few frames with [8] in `Stacks/Canales/<pos>_canales_BF.tif` and retrain it.
- **Bad frames:** if there are out-of-focus or defective frames, mark them with [6] in the macro and run again: step 3 is redone only for the positions whose `_cajas.zip` has changed.
- **Boxes corrected by hand:** likewise; when you save your boxes in the macro, step 3 redoes that position using your boxes in those frames (followed by steps 4 and 6).
- **End of the movie:** if something stops being valid towards the end (the medium runs out…), create `Stacks/Canales/fotos_validas.csv` (columns `imagen,ultima_foto,motivo`) and run again from step 3.